# Mitigação Prática da Fragmentação de Tópicos no BERTopic

Pipeline completo: coleta de artigos de Computação (arXiv), pré-processamento, embeddings (genérico vs SciBERT), modelagem (LDA, BERTopic+HDBSCAN, BERTopic+UMAP+K-Means) e métricas de validação (Cv, Taxa de Retenção, AMI).

Este notebook é pensado para rodar no **Google Colab**: a primeira célula clona o repositório e instala as dependências.

In [ ]:
import os

REPO_URL = "https://github.com/jeoaraujx/Projeto-de-pesquisa.git"
REPO_DIR = "Projeto-de-pesquisa"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm -q

### ⚠️ Reinicie o runtime agora

O Colab já vem com uma versão do NumPy carregada na memória antes desta célula rodar. Como o `requirements.txt` fixa `numpy==1.26.4` (necessário para compatibilidade com o `gensim`), instalar por cima do NumPy já carregado causa o erro `ValueError: numpy.dtype size changed, may indicate binary incompatibility`.

A célula abaixo reinicia o runtime automaticamente (o processo é encerrado e o Colab reconecta sozinho). **Depois que ela rodar, execute as células novamente a partir da seção "1. Coleta de dados"** — não precisa rodar a célula de instalação de novo, ela já fica em cache.

In [ ]:
import os
os.kill(os.getpid(), 9)

### Depois do restart, rode esta célula primeiro

O restart do runtime volta o diretório de trabalho para `/content`, perdendo o `%cd` da célula de instalação. Esta célula reentra na pasta do repositório antes de qualquer `import src...`.

In [ ]:
import os

REPO_DIR = "Projeto-de-pesquisa"
if os.path.basename(os.getcwd()) != REPO_DIR:
    os.chdir(f"/content/{REPO_DIR}")
print("Diretório atual:", os.getcwd())

## 1. Coleta de dados (arXiv)

Já existe um snapshot versionado em `data/raw/arxiv_cs_raw.jsonl`. Para recoletar do zero, rode a célula abaixo (leva alguns minutos por causa do rate-limit da API).

In [ ]:
# Descomente para recoletar do zero:
# from src.data_collection import collect_dataset, save_jsonl
# from pathlib import Path
# articles = collect_dataset()
# save_jsonl(articles, Path("data/raw/arxiv_cs_raw.jsonl"))

import pandas as pd

df_raw = pd.read_json("data/raw/arxiv_cs_raw.jsonl", lines=True)
print(f"Total de artigos coletados: {len(df_raw)}")
df_raw["primary_category"].value_counts()

## 2. Pré-processamento

In [ ]:
from pathlib import Path
from src.preprocessing import preprocess_corpus

df = preprocess_corpus(Path("data/raw/arxiv_cs_raw.jsonl"), Path("data/processed/corpus_clean.csv"))
df.head()

## 3. Pipeline completo (embeddings + LDA + BERTopic HDBSCAN + BERTopic UMAP/K-Means + métricas)

Executa `run_pipeline.py`, que gera `results/metrics_comparison.csv`, `results/topics_top_words.csv` e as figuras em `results/figures/`.

In [ ]:
!python run_pipeline.py

## 4. Resultados

In [ ]:
metrics_df = pd.read_csv("results/metrics_comparison.csv")
metrics_df

In [ ]:
from IPython.display import Image
Image("results/figures/metrics_comparison.png")

In [ ]:
Image("results/figures/elbow_curve.png")

In [ ]:
Image("results/figures/umap_scatter.png")

In [ ]:
topics_df = pd.read_csv("results/topics_top_words.csv")
topics_df